In [1]:
# CELL 1: Load all government schemes

import json
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\analysis\\all_schemes.json")

with open(INPUT_FILE, encoding="utf-8") as f:
    data = json.load(f)

schemes = data if isinstance(data, list) else data["schemes"]

print("Total schemes:", len(schemes))

Total schemes: 4428


In [2]:
# CELL 2: Configure recursive splitting as a fallback for long text

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def to_text(value):
    if value is None:
        return ""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def split_long_text(text):
    if len(text) <= 800:
        return [text]
    return splitter.split_text(text)

In [7]:
# CELL 3: Create feature-based parent and child documents

sections = [
    "brief_description",
    "details",
    "eligibility",
    "benefits",
    "exclusions",
    "application_process",
    "documents_required"
]

parents = []
children = []

for scheme in schemes:
    scheme_id = scheme["scheme_id"]

    base_metadata = {
        "scheme_id": scheme_id,
        "scheme_name": scheme.get("name", ""),
        "categories": scheme.get("categories", []),
        "source_url": scheme.get("myscheme_url", "")
    }

    # Process each scheme feature separately
    for section in sections:
        value = scheme.get(section)

        if not value:
            continue

        parent_id = f"{scheme_id}_{section.upper()}"

        parent_text = f"Scheme: {scheme.get('name', '')}\nSection: {section}\n{to_text(value)}"

        parents.append({
            "parent_id": parent_id,
            "text": parent_text,
            **base_metadata,
            "section": section
        })

        # Lists are split by their original items.
        # Long individual items use recursive splitting.
        items = value if isinstance(value, list) else [value]

        child_number = 1

        for item in items:
            item_text = to_text(item).strip()

            if not item_text:
                continue

            for part in split_long_text(item_text):
                children.append({
                    "child_id": f"{parent_id}_CHILD_{child_number:03d}",
                    "parent_id": parent_id,
                    "scheme_id": scheme_id,
                    "section": section,
                    "text": part,
                    "categories": base_metadata["categories"],
                    "document_type": "scheme_section"
                })
                child_number += 1

    # CELL: Keep complete FAQ as parent and split long answers into children

    FAQ_CHUNK_SIZE = 800
    FAQ_OVERLAP = 100

    for i, faq in enumerate(scheme.get("faqs", []), 1):

        if not isinstance(faq, dict):
            continue

        question = to_text(faq.get("question")).strip()
        answer = to_text(faq.get("answer")).strip()

        if not question or not answer:
            continue

        parent_id = f"{scheme_id}_FAQ_{i:03d}"

        faq_text = f"Question: {question}\nAnswer: {answer}"

        parents.append({
            "parent_id": parent_id,
            "text": faq_text,
            **base_metadata,
            "section": "faq"
        })

        # Keep short FAQs as one child
        if len(faq_text) <= FAQ_CHUNK_SIZE:
            answer_parts = [answer]
        else:
            answer_parts = splitter.split_text(answer)

        for j, part in enumerate(answer_parts, 1):
            children.append({
                "child_id": f"{parent_id}_CHILD_{j:03d}",
                "parent_id": parent_id,
                "scheme_id": scheme_id,
                "section": "faq",
                "text": f"Question: {question}\nAnswer: {part}",
                "categories": base_metadata["categories"],
                "document_type": "faq"
            })

print("Parent documents:", len(parents))
print("Child documents:", len(children))

Parent documents: 75982
Child documents: 122823


In [8]:
# CELL 4: Analyze and export structure-based chunks

parent_df = pd.DataFrame(parents)
child_df = pd.DataFrame(children)

child_df["characters"] = child_df["text"].str.len()

print("Unique schemes:", child_df["scheme_id"].nunique())
print("Unique parents:", child_df["parent_id"].nunique())
print("Duplicate child IDs:", child_df["child_id"].duplicated().sum())

print("\nChunks by section:")
print(child_df["section"].value_counts())

print("\nChunk length statistics:")
print(child_df["characters"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
).round(2))

OUTPUT = Path("data/processed/EXP_002_structure_based")
OUTPUT.mkdir(parents=True, exist_ok=True)

for name, records in [
    ("parents.jsonl", parents),
    ("children.jsonl", children)
]:
    with open(OUTPUT / name, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\nExperiment 002 saved successfully!")

Unique schemes: 4428
Unique parents: 75982
Duplicate child IDs: 0

Chunks by section:
section
faq                    48866
documents_required     36515
application_process    12532
benefits                6590
details                 6507
eligibility             6481
brief_description       4428
exclusions               904
Name: count, dtype: int64

Chunk length statistics:
count    122823.00
mean        245.70
std         211.26
min           1.00
50%         190.00
90%         609.00
95%         733.00
99%         797.00
max        1120.00
Name: characters, dtype: float64

Experiment 002 saved successfully!


In [5]:
# CELL 1: Inspect documents-required structure

for scheme in schemes[:]:
    value = scheme.get("documents_required")

    if isinstance(value, list) and len(value) > 10:
        print("Scheme:", scheme["name"])
        print("Number of items:", len(value))
        print("Sample items:", value[:5])
        break

Scheme: Stand-Up India
Number of items: 17
Sample items: ['Proof of Identity: Voter’s ID Card / Passport / Driving License / PAN Card / Signature identification from present bankers of proprietor, partner of director ( if a company)', 'Proof of Residence: Recent telephone bills, electricity bill, property tax receipt /Passport / voter’s ID Card of Proprietor, partner of Director (if a company)', 'Proof of Business Address', 'Proof that the applicant is not a defaulter in any Bank / Financial Institution', 'Memorandum and articles of association of the Company / Partnership Deed of partners etc.']


In [9]:
# CELL 2: Inspect chunks exceeding 800 characters

oversized = child_df[child_df["characters"] > 800]

print("Oversized chunks:", len(oversized))

display(
    oversized.sort_values("characters", ascending=False)[
        ["scheme_id", "parent_id", "section", "characters", "text"]
    ].head(10)
)

Oversized chunks: 109


,scheme_id,parent_id,section,characters,text
106971,TqKLzp0BXu-iUKXqllJX,TqKLzp0BXu-iUKXqllJX_FAQ_001,faq,1120,Question: Can An Existing PLI Beneficiary Comp...
106976,TqKLzp0BXu-iUKXqllJX,TqKLzp0BXu-iUKXqllJX_FAQ_005,faq,1082,Question: Can An Existing PLI Beneficiary Comp...
106974,TqKLzp0BXu-iUKXqllJX,TqKLzp0BXu-iUKXqllJX_FAQ_003,faq,1034,Question: If An Applicant Company At The Time ...
51850,B6KXGp4BXu-iUKXqnFTk,B6KXGp4BXu-iUKXqnFTk_FAQ_004,faq,979,Question: The scholarship document mentions sp...
16429,TMcMHJ4B0AGc0F0imBBj,TMcMHJ4B0AGc0F0imBBj_FAQ_019,faq,929,Question: What should be done for the unspent ...
79495,OaE9GJgBXu-iUKXqBnkY,OaE9GJgBXu-iUKXqBnkY_FAQ_013,faq,923,Question: What should be done for the unspent ...
120866,x6Kd5pwBXu-iUKXqgUuK,x6Kd5pwBXu-iUKXqgUuK_FAQ_023,faq,916,Question: What is the process of fund disburse...
86809,CKJ3hZ0BXu-iUKXqA1Am,CKJ3hZ0BXu-iUKXqA1Am_FAQ_006,faq,912,Question: What are the mandatory documents to ...
96744,acbkF5gB0AGc0F0iRDdA,acbkF5gB0AGc0F0iRDdA_FAQ_006,faq,911,Question: Which services are provided under pa...
88950,pBe5F5gB5uF80_MGX5A2,pBe5F5gB5uF80_MGX5A2_FAQ_005,faq,910,Question: Give brief information about what do...


### EXP_002_v2

We'll make two improvements:
1. Group related documents_required items into chunks.
2. Keep complete FAQs as parent documents and split only exceptionally long answers.
We'll keep all original wording and preserve item order.

In [11]:
# CELL 1: Configure chunk sizes and helper functions

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

def group_items(items, max_chars=800):
    """Group list items without exceeding the preferred character budget."""
    groups = []
    current = ""

    for item in items:
        text = to_text(item).strip()
        if not text:
            continue

        candidate = f"{current}\n{text}" if current else text

        if len(candidate) > max_chars and current:
            groups.append(current)
            current = text
        else:
            current = candidate

    if current:
        groups.append(current)

    return groups

In [12]:
# CELL 2: Group document requirements instead of creating tiny chunks

document_children = []

for scheme in schemes:
    scheme_id = scheme["scheme_id"]
    value = scheme.get("documents_required")

    if not value:
        continue

    parent_id = f"{scheme_id}_DOCUMENTS_REQUIRED"

    items = value if isinstance(value, list) else [value]

    groups = group_items(items, max_chars=800)

    child_number = 1

    for group in groups:
        for part in split_long_text(group):
            document_children.append({
                "child_id": f"{parent_id}_CHILD_{child_number:03d}",
                "parent_id": parent_id,
                "scheme_id": scheme_id,
                "section": "documents_required",
                "text": part,
                "document_type": "scheme_section"
            })
            child_number += 1

print("Document requirement chunks:", len(document_children))

Document requirement chunks: 5481


In [13]:
# CELL 3: Keep FAQs atomic unless their answers are exceptionally long

faq_children = []

for scheme in schemes:
    scheme_id = scheme["scheme_id"]

    for i, faq in enumerate(scheme.get("faqs", []), 1):
        if not isinstance(faq, dict):
            continue

        question = to_text(faq.get("question")).strip()
        answer = to_text(faq.get("answer")).strip()

        if not question or not answer:
            continue

        parent_id = f"{scheme_id}_FAQ_{i:03d}"

        if len(answer) <= 800:
            answer_parts = [answer]
        else:
            answer_parts = splitter.split_text(answer)

        for j, part in enumerate(answer_parts, 1):
            faq_children.append({
                "child_id": f"{parent_id}_CHILD_{j:03d}",
                "parent_id": parent_id,
                "scheme_id": scheme_id,
                "section": "faq",
                "text": f"Question: {question}\nAnswer: {part}",
                "document_type": "faq"
            })

print("FAQ child chunks:", len(faq_children))

FAQ child chunks: 48866


In [16]:
# CELL 3: Keep FAQs atomic unless their answers are exceptionally long

faq_children = []

for scheme in schemes:
    scheme_id = scheme["scheme_id"]

    for i, faq in enumerate(scheme.get("faqs", []), 1):
        if not isinstance(faq, dict):
            continue

        question = to_text(faq.get("question")).strip()
        answer = to_text(faq.get("answer")).strip()

        if not question or not answer:
            continue

        parent_id = f"{scheme_id}_FAQ_{i:03d}"

        # Reserve space for the question and answer prefix
        prefix = f"Question: {question}\nAnswer: "
        max_answer_chars = 800 - len(prefix)

        if max_answer_chars <= 0:
            print(f"Warning: Question exceeds 800 characters: {parent_id}")
            continue

        # Keep short FAQs atomic; split exceptionally long answers
        if len(answer) <= max_answer_chars:
            answer_parts = [answer]
        else:
            faq_splitter = RecursiveCharacterTextSplitter(
                chunk_size=max_answer_chars,
                chunk_overlap=min(100, max_answer_chars // 4),
                separators=["\n\n", "\n", ". ", " ", ""]
            )
            answer_parts = faq_splitter.split_text(answer)

        for j, part in enumerate(answer_parts, 1):
            faq_children.append({
                "child_id": f"{parent_id}_CHILD_{j:03d}",
                "parent_id": parent_id,
                "scheme_id": scheme_id,
                "section": "faq",
                "text": prefix + part,
                "document_type": "faq"
            })

print("FAQ child chunks:", len(faq_children))

FAQ child chunks: 48935


In [17]:
# CELL 4: Combine optimized children and save EXP_002_v2

# Remove old document-required and FAQ children
other_children = [
    child for child in children
    if child["section"] not in ["documents_required", "faq"]
]

optimized_children = (
    other_children +
    document_children +
    faq_children
)

child_df_v2 = pd.DataFrame(optimized_children)
child_df_v2["characters"] = child_df_v2["text"].str.len()

print("Total child chunks:", len(child_df_v2))
print("\nChunks by section:")
print(child_df_v2["section"].value_counts())

print("\nChunk length statistics:")
print(child_df_v2["characters"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
).round(2))

OUTPUT = Path("data/processed/EXP_002_v2")
OUTPUT.mkdir(parents=True, exist_ok=True)

with open(OUTPUT / "children.jsonl", "w", encoding="utf-8") as f:
    for record in optimized_children:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\nEXP_002_v2 saved successfully!")

Total child chunks: 91858

Chunks by section:
section
faq                    48935
application_process    12532
benefits                6590
details                 6507
eligibility             6481
documents_required      5481
brief_description       4428
exclusions               904
Name: count, dtype: int64

Chunk length statistics:
count    91858.00
mean       328.97
std        204.14
min          1.00
50%        260.00
90%        693.00
95%        765.00
99%        797.00
max        800.00
Name: characters, dtype: float64

EXP_002_v2 saved successfully!


### 1. Experiment comparison
EXP_002
#### 122,823
Child chunks

EXP_002_v2
#### 91,789
Child chunks


#### 25.27%
Fewer chunks than EXP_002.

In [18]:
# CELL 1: Validate optimized parent-child relationships and oversized chunks

parent_ids = {p["parent_id"] for p in parents}

missing_parents = [
    c for c in optimized_children
    if c["parent_id"] not in parent_ids
]

duplicate_ids = child_df_v2["child_id"].duplicated().sum()

oversized = child_df_v2[child_df_v2["characters"] > 800]

print("Missing parent references:", len(missing_parents))
print("Duplicate child IDs:", duplicate_ids)
print("Chunks above 800 characters:", len(oversized))

print("\nOversized chunks by section:")
print(oversized["section"].value_counts())

print("\nLargest chunks:")
display(
    oversized.nlargest(5, "characters")[
        ["scheme_id", "section", "characters", "text"]
    ]
)

Missing parent references: 0
Duplicate child IDs: 0
Chunks above 800 characters: 0

Oversized chunks by section:
Series([], Name: count, dtype: int64)

Largest chunks:


,scheme_id,section,characters,text


In [19]:
import json
from pathlib import Path

output_path = Path("data/processed/EXP_002_v2")
output_path.mkdir(parents=True, exist_ok=True)

with open(output_path / "children.jsonl", "w", encoding="utf-8") as f:
    for child in optimized_children:
        f.write(json.dumps(child, ensure_ascii=False) + "\n")

print("EXP_002_v2 saved successfully.")

EXP_002_v2 saved successfully.
